In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='feature_embedding/mol')
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'feature_embedding/mol')



In [5]:
from __future__ import annotations

import gc
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator


In [6]:
source_csv = Path(_kp('dataset/KIBA_unseen_LUDe_augment_sim08.csv'))
decoder_path = Path(_kp('feature_embedding/mol/KIBA_unseen_LUDe_augmented_sim08_LSAE_of1024d_decoder_v4.pt'))
embedding_output = Path(_kp('dataset/KIBA_unseen_LUDe_augmented_sim08_LSAE_latent_top95_features_v4.csv'))

print(f"Input:  {source_csv}")
print(f"Output: {embedding_output}")

Input:  /home/s1856122/KinSPEC_final/dataset/KIBA_unseen_active_filter_LUDe_augment_sim08.csv
Output: /home/s1856122/KinSPEC_final/dataset/KIBA_unseen_active_filter_LUDe_augmented_sim08_LSAE_latent_top95_features_v4.csv


In [7]:
# Memory controls:
# - read_chunk_rows controls CPU RAM usage.
# - batch_size in SAEConfig controls GPU memory usage.
# Morgan fingerprints are generated from SMILES in memory, never saved to disk.
READ_CHUNK_ROWS = 5_000
GPU_BATCH_SIZE = 1_024

In [8]:
@dataclass(frozen=True)
class SAEConfig:
    device: torch.device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    input_dim: int = 4096
    latent_dim: int = 1024
    selected_latent_dim: int = 417
    read_chunk_rows: int = READ_CHUNK_ROWS
    batch_size: int = GPU_BATCH_SIZE
    source_csv: Path = source_csv
    output_csv: Path = embedding_output
    decoder_path: Path = decoder_path
    model_path: Path = Path(_kp('feature_embedding/mol/LSAE_chembl_1900k_full_dataset_v4_model.pt'))
    idx_path: Path = Path(_kp('feature_embedding/mol/LSAE_chembl_1900k_full_dataset_v4_topk_of1024d_top99.pt'))


class SparseLinearAE(nn.Module):
    def __init__(self, cfg: SAEConfig):
        super().__init__()
        self.encoder = nn.Linear(cfg.input_dim, cfg.latent_dim, bias=True)
        self.decoder = nn.Linear(cfg.latent_dim, cfg.input_dim, bias=True)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


def load_model_and_idx(cfg: SAEConfig):
    print(f"[info] Using device: {cfg.device}")

    model = SparseLinearAE(cfg).to(cfg.device)
    checkpoint = torch.load(
        cfg.model_path,
        map_location=cfg.device,
        weights_only=True,
    )
    model.load_state_dict(checkpoint)
    model.eval()

    keep_idx = torch.load(
        cfg.idx_path,
        map_location=cfg.device,
        weights_only=True,
    ).long()[: cfg.selected_latent_dim]

    if keep_idx.numel() != cfg.selected_latent_dim:
        raise ValueError(
            f"Expected {cfg.selected_latent_dim} latent indices, "
            f"but found {keep_idx.numel()}"
        )

    print(
        f"[info] Loaded model and {keep_idx.numel()} latent indices "
        f"on {cfg.device}"
    )
    return model, keep_idx


def smiles_to_morgan_uint8(
    series: pd.Series,
    generator,
    input_dim: int,
) -> np.ndarray:
    """Generate binary Morgan fingerprints in input order, without saving them."""
    bits = np.zeros((len(series), input_dim), dtype=np.uint8)
    for position, (row_index, smiles) in enumerate(series.items()):
        # Chunk indices retain the zero-based data row in the source CSV.
        if pd.isna(smiles) or not str(smiles).strip():
            raise ValueError(f"Missing SMILES at source data row {row_index + 1}")

        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None or mol.GetNumAtoms() == 0:
            raise ValueError(
                f"Invalid SMILES at source data row {row_index + 1}: {smiles!r}"
            )

        fingerprint = generator.GetFingerprint(mol)
        DataStructs.ConvertToNumpyArray(fingerprint, bits[position])

    return bits


def save_decoder_subset(
    model: SparseLinearAE,
    keep_idx: torch.Tensor,
    output_path: Path,
):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    decoder_data = {
        "idx": keep_idx.detach().cpu(),
        "decoder_sub": model.decoder.weight[:, keep_idx].detach().cpu(),
        "bias": model.decoder.bias.detach().cpu(),
    }
    torch.save(decoder_data, output_path)
    print(f"[save] Decoder submatrix -> {output_path}")


def extract_and_save_streaming(cfg: SAEConfig):
    """Stream SMILES, generate Morgan bits in memory, and save embeddings."""
    if not cfg.source_csv.exists():
        raise FileNotFoundError(cfg.source_csv)

    cfg.output_csv.parent.mkdir(parents=True, exist_ok=True)
    temporary_output = cfg.output_csv.with_name(cfg.output_csv.name + ".part")

    model, keep_idx = load_model_and_idx(cfg)

    # Compute only the selected latent dimensions instead of all 1024 outputs.
    selected_weight = model.encoder.weight.index_select(0, keep_idx).detach()
    selected_bias = model.encoder.bias.index_select(0, keep_idx).detach()

    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=2,
        fpSize=cfg.input_dim,
        includeChirality=False,
    )

    reader = pd.read_csv(
        cfg.source_csv,
        usecols=["SMILES"],
        dtype={"SMILES": "string"},
        chunksize=cfg.read_chunk_rows,
    )

    write_header = True
    rows_written = 0

    try:
        with temporary_output.open("w", newline="") as output_file:
            with tqdm(desc="Embedding Morgan fingerprints", unit="rows") as progress:
                for chunk in reader:
                    bits = smiles_to_morgan_uint8(
                        chunk["SMILES"],
                        generator=generator,
                        input_dim=cfg.input_dim,
                    )

                    for start in range(0, len(bits), cfg.batch_size):
                        batch = np.ascontiguousarray(
                            bits[start : start + cfg.batch_size]
                        )
                        xb = torch.from_numpy(batch).to(
                            cfg.device,
                            dtype=torch.float32,
                        )

                        with torch.inference_mode():
                            latent = F.linear(
                                xb,
                                selected_weight,
                                selected_bias,
                            )

                        latent_cpu = latent.cpu().numpy()
                        pd.DataFrame(latent_cpu).to_csv(
                            output_file,
                            index=False,
                            header=write_header,
                        )
                        write_header = False
                        rows_written += len(latent_cpu)

                        del batch, xb, latent, latent_cpu

                    progress.update(len(chunk))
                    del bits, chunk
                    gc.collect()

        temporary_output.replace(cfg.output_csv)
        save_decoder_subset(model, keep_idx, cfg.decoder_path)

    except Exception:
        print(f"[error] Partial output retained at: {temporary_output}")
        raise

    finally:
        del selected_weight, selected_bias, model, keep_idx
        if cfg.device.type == "cuda":
            torch.cuda.empty_cache()

    print(f"[save] {rows_written:,} embeddings -> {cfg.output_csv}")


def main():
    cfg = SAEConfig()

    if cfg.device.type == "cuda":
        properties = torch.cuda.get_device_properties(0)
        print(f"[info] GPU: {torch.cuda.get_device_name(0)}")
        print(f"[info] GPU memory: {properties.total_memory / 1e9:.1f} GB")
    else:
        print("[warning] CUDA is unavailable; processing will use CPU")

    print(f"[info] CPU read chunk: {cfg.read_chunk_rows:,} rows")
    print(f"[info] GPU batch: {cfg.batch_size:,} rows")
    extract_and_save_streaming(cfg)


if __name__ == "__main__":
    main()

[info] GPU: NVIDIA GeForce RTX 5090
[info] GPU memory: 33.7 GB
[info] CPU read chunk: 5,000 rows
[info] GPU batch: 1,024 rows
[info] Using device: cuda
[info] Loaded model and 417 latent indices on cuda


Embedding Morgan fingerprints: 0rows [00:00, ?rows/s]

Embedding Morgan fingerprints: 19608rows [00:04, 4326.81rows/s]

[save] Decoder submatrix -> KIBA_unseen_active_filter_LUDe_augmented_sim08_LSAE_of1024d_decoder_v4.pt
[save] 19,608 embeddings -> /home/s1856122/KinSPEC_final/dataset/KIBA_unseen_active_filter_LUDe_augmented_sim08_LSAE_latent_top95_features_v4.csv
